In [2]:
# ==============================
# 1. IMPORT LIBRARIES
# ==============================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, recall_score, roc_auc_score, classification_report

from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42


# ==============================
# 2. LOAD DATA
# ==============================
df = pd.read_csv("/content/drive/MyDrive/diabetes.csv")

# ==============================
# 3. DATA CLEANING
# ==============================
cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

for col in cols:
    df[col] = df[col].replace(0, np.nan)
    df[col].fillna(df[col].median(), inplace=True)

X = df.drop('Outcome', axis=1)
y = df['Outcome']

# ==============================
# 4. TRAIN-TEST SPLIT
# ==============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

# ==============================
# 5. CROSS VALIDATION STRATEGY
# ==============================
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# ==============================
# 6. DEFINE PIPELINES + PARAMS
# ==============================

# Logistic Regression
pipe_log = ImbPipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=RANDOM_STATE)),
    ('model', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

param_log = {
    'model__C': [0.01, 0.1, 1, 10]
}

# Random Forest
pipe_rf = ImbPipeline([
    ('smote', SMOTE(random_state=RANDOM_STATE)),
    ('model', RandomForestClassifier(random_state=RANDOM_STATE))
])

param_rf = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [5, 10]
}

# SVM
pipe_svm = ImbPipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=RANDOM_STATE)),
    ('model', SVC(probability=True, random_state=RANDOM_STATE))
])

param_svm = {
    'model__C': [0.1, 1, 10],
    'model__kernel': ['rbf']
}

# KNN
pipe_knn = ImbPipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=RANDOM_STATE)),
    ('model', KNeighborsClassifier())
])

param_knn = {
    'model__n_neighbors': [3, 5, 7]
}

# Decision Tree
pipe_dt = ImbPipeline([
    ('smote', SMOTE(random_state=RANDOM_STATE)),
    ('model', DecisionTreeClassifier(random_state=RANDOM_STATE))
])

param_dt = {
    'model__max_depth': [3, 5, 10]
}

# XGBoost
pipe_xgb = ImbPipeline([
    ('smote', SMOTE(random_state=RANDOM_STATE)),
    ('model', XGBClassifier(eval_metric='logloss', random_state=RANDOM_STATE))
])

param_xgb = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [3, 5],
    'model__learning_rate': [0.01, 0.1]
}

# LightGBM
pipe_lgbm = ImbPipeline([
    ('smote', SMOTE(random_state=RANDOM_STATE)),
    ('model', LGBMClassifier(random_state=RANDOM_STATE))
])

param_lgbm = {
    'model__n_estimators': [100, 200],
    'model__learning_rate': [0.01, 0.1],
    'model__num_leaves': [31, 50]
}

# ==============================
# 7. TRAIN + TUNE MODELS
# ==============================
def train_model(name, pipeline, params):
    grid = GridSearchCV(
        pipeline,
        params,
        cv=cv,
        scoring='f1',
        n_jobs=-1
    )
    grid.fit(X_train, y_train)
    print(f"{name} Best Params:", grid.best_params_)
    return grid.best_estimator_

log_model = train_model("Logistic", pipe_log, param_log)
rf_model = train_model("Random Forest", pipe_rf, param_rf)
svm_model = train_model("SVM", pipe_svm, param_svm)
knn_model = train_model("KNN", pipe_knn, param_knn)
dt_model = train_model("Decision Tree", pipe_dt, param_dt)
xgb_model = train_model("XGBoost", pipe_xgb, param_xgb)
lgbm_model = train_model("LightGBM", pipe_lgbm, param_lgbm)

# Naive Bayes (no tuning)
nb_model = ImbPipeline([
    ('scaler', StandardScaler()),
    ('model', GaussianNB())
])
nb_model.fit(X_train, y_train)

# ==============================
# 8. EVALUATION
# ==============================
models = {
    "Logistic": log_model,
    "Random Forest": rf_model,
    "SVM": svm_model,
    "KNN": knn_model,
    "Decision Tree": dt_model,
    "Naive Bayes": nb_model,
    "XGBoost": xgb_model,
    "LightGBM": lgbm_model
}

results = []

for name, model in models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_prob)

    results.append([name, acc, f1, recall, roc])

    print(f"\n{name}")
    print("Accuracy:", acc)
    print("F1:", f1)
    print("Recall:", recall)
    print("ROC-AUC:", roc)
    print(classification_report(y_test, y_pred))

# ==============================
# 9. FINAL COMPARISON
# ==============================
results_df = pd.DataFrame(results, columns=["Model", "Accuracy", "F1", "Recall", "ROC-AUC"])
print("\nFinal Model Comparison:")
print(results_df.sort_values(by="F1", ascending=False))

Logistic Best Params: {'model__C': 10}
Random Forest Best Params: {'model__max_depth': 5, 'model__n_estimators': 100}
SVM Best Params: {'model__C': 1, 'model__kernel': 'rbf'}
KNN Best Params: {'model__n_neighbors': 7}
Decision Tree Best Params: {'model__max_depth': 3}
XGBoost Best Params: {'model__learning_rate': 0.01, 'model__max_depth': 3, 'model__n_estimators': 100}
[LightGBM] [Info] Number of positive: 400, number of negative: 400
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000148 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1192
[LightGBM] [Info] Number of data points in the train set: 800, number of used features: 8
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with po